# Dataset Acquisition

## Scientific objective
Acquire unmodified hERG, Ames, and selected Tox21 source tables through documented access paths; compute checksums and registry entries.

## Inputs
- `configs/data_config.yaml`
- Internet access
- Pinned `pytdc` for hERG/Ames

## Expected outputs
- Raw files under `data/raw/{herg,ames,tox21}`
- `data/metadata/dataset_registry.csv`
- Explicit issue report if access fails

## Dependencies
requests, pandas, pytdc

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
TDC tables are treated as downloaded ML-ready distributions, not primary assay records. The Tox21 mirror is checked against the official NCATS endpoint documentation.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Dataset availability, access paths, and terms can change. Ambiguous licenses are recorded and must be cleared before redistribution.

## Next notebook
[03_data_provenance_and_endpoint_definitions.ipynb](./03_data_provenance_and_endpoint_definitions.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723}


In [2]:
from toxicity_screening.pipeline import acquire_datasets
registry = acquire_datasets(ROOT)
display(registry)

Downloading...
100%|██████████| 885k/885k [00:00<00:00, 1.50MiB/s]
Loading...
Done!
Downloading...
100%|██████████| 344k/344k [00:05<00:00, 65.3kiB/s] 
Loading...
Done!


,dataset_name,endpoint,source,official_url,version,download_date,license,citation,raw_filename,checksum,number_of_records,label_definition,measurement_units,assay_context,notes,label_columns,acquisition_method,download_url
0,hERG_Karim,herg_blockade,Therapeutics Data Commons,https://tdcommons.ai/single_pred_tasks/tox/,pytdc-1.1.15 snapshot at acquisition,2026-07-25T21:27:17.648989+00:00,TDC page is ambiguous: it displays 'Not Specif...,Karim et al. CardioTox net. Journal of Cheminf...,data\raw\herg\herg_karim_tdc.csv,554f5f6db861de0c887c72e267fa5124e732536d126880...,13445,1 if hERG blocker with reported IC50 < 10 uM; ...,uM,Integrated literature/database classification;...,Unmodified table returned by pinned pytdc; not...,"[""Y""]",pytdc,
1,AMES,ames_mutagenicity,Therapeutics Data Commons,https://tdcommons.ai/single_pred_tasks/tox/,pytdc-1.1.15 snapshot at acquisition,2026-07-25T21:27:24.179030+00:00,TDC page is ambiguous: it displays 'Not Specif...,Xu et al. In silico prediction of chemical Ame...,data\raw\ames\ames_tdc.csv,f4a0b36b82c0a7dd03306d6a1b0b455cb28d1784fba876...,7278,1 mutagenic; 0 non-mutagenic in the aggregated...,binary,Aggregated from four publications; strain and ...,Unmodified table returned by pinned pytdc; not...,"[""Y""]",pytdc,
2,Tox21_MoleculeNet,tox21_multilabel,NCATS Tox21 Challenge via DeepChem/MoleculeNet...,https://tripod.nih.gov/tox21/challenge/data.jsp,2014 challenge training data / current DeepChe...,2026-07-25T21:27:25.821572+00:00,No explicit dataset-specific license was locat...,Tox21 Data Challenge 2014; MoleculeNet: Wu et ...,data\raw\tox21\tox21.csv.gz,45d09792492ce049039dd24aa27b07fc79ce20c573187d...,7831,1 active; 0 inactive; blank missing/unavailable,binary assay activity,High-throughput in vitro stress-response assays,Downloaded directly from URL,"[""SR-p53"", ""SR-ATAD5"", ""SR-ARE"", ""SR-MMP""]",url,https://deepchemdata.s3-us-west-1.amazonaws.co...


In [3]:
required_columns = set(pd.read_csv(ROOT / "data/metadata/dataset_registry_schema.csv").columns)
missing = required_columns - set(registry.columns)
assert not missing, f"Registry columns missing: {sorted(missing)}"
assert registry["checksum"].str.fullmatch(r"[0-9a-f]{64}").all()
assert (registry["number_of_records"] > 0).all()

### Completion gate
Confirm that the declared artifacts exist before continuing to `03_data_provenance_and_endpoint_definitions.ipynb`.